<a href="https://colab.research.google.com/github/davidmiheev/ml/blob/main/yandex_coderun_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 184. Восстановление коэффициентов

https://coderun.yandex.ru/problem/coefficients-restoration/description

In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import os
import zipfile
from typing import List

def download_data() -> List[str]:
    """
    Downloads data from a Yandex Disk public link.

    It first attempts to download a single file if the filename is specified
    in the response headers. If not, it saves the content and assumes it's a
    zip archive. If the zip assumption fails, it then inspects the content
    to treat it as a CSV or TXT file.

    Returns:
        List[str]: A list of file paths for the downloaded/extracted files,
                   or an empty list if an error occurs.
    """
    public_key = "https://yadi.sk/d/RXnmNGp4PZnMrg" # Replace with the correct public key if needed
    base_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key="

    try:
        # Get the download link
        final_url = base_url + public_key
        response = requests.get(final_url)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
        download_info = response.json()

        if 'href' in download_info:
            download_url = download_info['href']

            # Download the file(s)
            download_response = requests.get(download_url, stream=True)
            download_response.raise_for_status()

            # Check for 'content-disposition' header for a single file name
            content_disposition = download_response.headers.get('content-disposition')
            if content_disposition and 'filename=' in content_disposition:
                filename = content_disposition.split('filename=')[1].strip('"')
                file_path = filename
                with open(file_path, 'wb') as f:
                    for chunk in download_response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded single file: {filename}")
                return [file_path] # Return a list with the single file path
            else:
                # If no filename, save to a temporary file first without an extension
                temp_file_path = 'downloaded_data.tmp'
                with open(temp_file_path, 'wb') as f:
                     for chunk in download_response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded content to temporary file: {temp_file_path}")

                # First, try to handle it as a zip file
                try:
                    with zipfile.ZipFile(temp_file_path, 'r') as zip_ref:
                        extracted_files = zip_ref.namelist()
                        zip_ref.extractall('.')
                        print(f"Successfully extracted {len(extracted_files)} files from archive.")
                    os.remove(temp_file_path) # Remove the temp file after extraction
                    print(f"Removed temporary archive: {temp_file_path}")
                    return extracted_files

                # --- NEW LOGIC: Fallback for non-zip files ---
                except zipfile.BadZipFile:
                    print(f"'{temp_file_path}' is not a zip file. Attempting to handle as a text file.")

                    # Inspect the first 1024 bytes to guess the file type
                    new_extension = '.txt' # Default to .txt
                    try:
                        with open(temp_file_path, 'r', encoding='utf-8') as f:
                            # A simple heuristic: if a comma is in the first line, it's likely a CSV
                            first_line = f.readline()
                            if ',' in first_line:
                                new_extension = '.csv'
                    except (UnicodeDecodeError, IOError) as e:
                        print(f"Could not read file as text to determine type: {e}. Defaulting to .txt")

                    # Rename the file with the determined extension
                    final_filename = 'downloaded_file' + new_extension
                    if os.path.exists(final_filename):
                        os.remove(final_filename) # Remove if it already exists to avoid errors
                    os.rename(temp_file_path, final_filename)

                    print(f"Handled download as a single file and renamed to: {final_filename}")
                    return [final_filename]

                except Exception as e:
                    print(f"An unexpected error occurred during file handling: {e}")
                    if os.path.exists(temp_file_path):
                        os.remove(temp_file_path) # Clean up on other errors
                    return []

        elif 'error' in download_info:
            print(f"Error from Yandex Disk API: {download_info['message']}")
            return []
        else:
            print("Unexpected response format from Yandex Disk API.")
            return []

    except requests.exceptions.RequestException as e:
        print(f"Error during network request: {e}")
        return []

# Example of how to use the function
if __name__ == '__main__':
    downloaded_files = download_data()
    if downloaded_files:
        print("\n--- Function finished ---")
        print(f"Final file paths returned: {downloaded_files}")
        # You can now proceed to load the data, e.g.:
        # if downloaded_files[0].endswith('.csv'):
        #     df = pd.read_csv(downloaded_files[0])
        #     print("\nSuccessfully loaded data into pandas DataFrame:")
        #     print(df.head())
    else:
        print("\n--- Function finished ---")
        print("Could not retrieve any files.")

Downloaded content to temporary file: downloaded_data.tmp
'downloaded_data.tmp' is not a zip file. Attempting to handle as a text file.
Handled download as a single file and renamed to: downloaded_file.csv

--- Function finished ---
Final file paths returned: ['downloaded_file.csv']


In [ ]:
import sys


def main():
    from math import sin, cos, tan, sqrt, log
    from sklearn.linear_model import LinearRegression
    import numpy as np
    import pandas as pd

    data = pd.read_csv('downloaded_file.csv')
    n = data.shape[0]
    X, y = [], []
    for i in range(n):
        x, f_x = map(float, data.iloc[i])
        X.append([sin(x) * sin(x), 2 * sin(x) * log(x), log(x) * log(x), x * x])
        y.append(f_x)

    X = np.array(X)
    y = np.array(y)
    model = LinearRegression(fit_intercept=False)
    model.fit(X, y)
    a2, ab, b2, c = model.coef_
    print(f'{sqrt(a2):.2f} {sqrt(b2):.2f} {c:.2f}')

if __name__ == '__main__':
    main()

3.14 2.72 4.00


## 177. Тайна египетских пирамид

https://coderun.yandex.ru/problem/egypt

In [ ]:
import sys


def main():
    from math import sin, cos, tan, sqrt
    from sklearn.linear_model import LinearRegression

    n = int(input())
    X, y = [], []
    for i in range(n):
        x, f_x = map(float, input().split())
        X.append([sin(x), sin(x) * sin(2 * x) / 2, cos(x) * sin(2 * x), cos(x) ** 3, cos(x) * sqrt(x)])
        y.append(f_x * cos(x))

    model = LinearRegression(fit_intercept=False)
    model.fit(X, y)
    a, b2, bc, c2, d = model.coef_
    print(f'{a:.2f} {sqrt(b2):.2f} {sqrt(c2):.2f} {d:.2f}')

if __name__ == '__main__':
    main()

## 156. Опечатки

https://coderun.yandex.ru/problem/typos

In [ ]:
import sys


def main():
    from collections import defaultdict, deque

    target = tuple(input())
    start = tuple(input())
    n = int(input())
    graph = defaultdict(set)
    for _ in range(n):
        a, b = input().split()
        graph[tuple(b)].add(tuple(a))

    seen = set([start])
    q = deque([(start, 0)])
    min_dist = -1
    while q:
        word, dist = q.popleft()

        if word == target:
            min_dist = dist
            break

        if dist == 4:
            continue

        for b in graph:
            for i in range(len(word) - len(b) + 1):
                if word[i:i + len(b)] == b:
                    for a in graph[b]:
                        new_word = word[:i] + a + word[i + len(b):]
                        if new_word not in seen:
                            q.append((new_word, dist + 1))
                            seen.add(new_word)

    print(min_dist)



if __name__ == '__main__':
    main()

## 548. Исправить все опечатки

https://coderun.yandex.ru/problem/fix-all-misprints

In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import os
import zipfile
from typing import List

def download_data() -> List[str]:
    """
    Downloads data from a Yandex Disk public link.

    It first attempts to download a single file if the filename is specified
    in the response headers. If not, it saves the content and assumes it's a
    zip archive. If the zip assumption fails, it then inspects the content
    to treat it as a CSV or TXT file.

    Returns:
        List[str]: A list of file paths for the downloaded/extracted files,
                   or an empty list if an error occurs.
    """
    public_key = "https://disk.yandex.ru/d/WfcBWhNQ0aH0Pw" # Replace with the correct public key if needed
    base_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key="

    try:
        # Get the download link
        final_url = base_url + public_key
        response = requests.get(final_url)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
        download_info = response.json()

        if 'href' in download_info:
            download_url = download_info['href']

            # Download the file(s)
            download_response = requests.get(download_url, stream=True)
            download_response.raise_for_status()

            # Check for 'content-disposition' header for a single file name
            content_disposition = download_response.headers.get('content-disposition')
            if content_disposition and 'filename=' in content_disposition:
                filename = content_disposition.split('filename=')[1].strip('"')
                file_path = filename
                with open(file_path, 'wb') as f:
                    for chunk in download_response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded single file: {filename}")
                return [file_path] # Return a list with the single file path
            else:
                # If no filename, save to a temporary file first without an extension
                temp_file_path = 'downloaded_data.tmp'
                with open(temp_file_path, 'wb') as f:
                     for chunk in download_response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded content to temporary file: {temp_file_path}")

                # First, try to handle it as a zip file
                try:
                    with zipfile.ZipFile(temp_file_path, 'r') as zip_ref:
                        extracted_files = zip_ref.namelist()
                        zip_ref.extractall('.')
                        print(f"Successfully extracted {len(extracted_files)} files from archive.")
                    os.remove(temp_file_path) # Remove the temp file after extraction
                    print(f"Removed temporary archive: {temp_file_path}")
                    return extracted_files

                # --- NEW LOGIC: Fallback for non-zip files ---
                except zipfile.BadZipFile:
                    print(f"'{temp_file_path}' is not a zip file. Attempting to handle as a text file.")

                    # Inspect the first 1024 bytes to guess the file type
                    new_extension = '.txt' # Default to .txt
                    try:
                        with open(temp_file_path, 'r', encoding='utf-8') as f:
                            # A simple heuristic: if a comma is in the first line, it's likely a CSV
                            first_line = f.readline()
                            if ',' in first_line:
                                new_extension = '.csv'
                    except (UnicodeDecodeError, IOError) as e:
                        print(f"Could not read file as text to determine type: {e}. Defaulting to .txt")

                    # Rename the file with the determined extension
                    final_filename = 'downloaded_file' + new_extension
                    if os.path.exists(final_filename):
                        os.remove(final_filename) # Remove if it already exists to avoid errors
                    os.rename(temp_file_path, final_filename)

                    print(f"Handled download as a single file and renamed to: {final_filename}")
                    return [final_filename]

                except Exception as e:
                    print(f"An unexpected error occurred during file handling: {e}")
                    if os.path.exists(temp_file_path):
                        os.remove(temp_file_path) # Clean up on other errors
                    return []

        elif 'error' in download_info:
            print(f"Error from Yandex Disk API: {download_info['message']}")
            return []
        else:
            print("Unexpected response format from Yandex Disk API.")
            return []

    except requests.exceptions.RequestException as e:
        print(f"Error during network request: {e}")
        return []

# Example of how to use the function
if __name__ == '__main__':
    downloaded_files = download_data()
    if downloaded_files:
        print("\n--- Function finished ---")
        print(f"Final file paths returned: {downloaded_files}")
        # You can now proceed to load the data, e.g.:
        # if downloaded_files[0].endswith('.csv'):
        #     df = pd.read_csv(downloaded_files[0])
        #     print("\nSuccessfully loaded data into pandas DataFrame:")
        #     print(df.head())
    else:
        print("\n--- Function finished ---")
        print("Could not retrieve any files.")

In [ ]:
from collections import defaultdict
import os

# --- Optimized Helper Functions ---

def edits1(word):
    """
    Creates a set of all words that are one edit away from the input word.
    """
    letters    = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
    replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
    inserts    = [L + c + R               for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def find_correction(word, dictionary):
    """
    Finds a correction for a word without using a pre-computed graph.
    It checks for corrections 1 edit away, and then 2 edits away.
    """
    # --- Level 1: Check for corrections 1 edit away ---
    one_edit_words = edits1(word)
    for w1 in one_edit_words:
        if w1 in dictionary:
            # Found a 1-edit correction.
            return [word, w1]

    # --- Level 2: Check for corrections 2 edits away (OPTIMIZED) ---
    # This is much faster than the previous nested loop.
    for w1 in one_edit_words:
        # Use a fast set intersection to find valid corrections.
        known_corrections = edits1(w1).intersection(dictionary)
        if known_corrections:
            # Found a valid 2-edit correction. Return the path.
            return [word, w1, known_corrections.pop()]

    # If no correction is found within 2 edits, return None.
    return None


# --- Optimized Main Script ---

# Create a dummy queries.txt if it doesn't exist for demonstration
if not os.path.exists('queries.txt'):
    print("Creating a dummy 'queries.txt' file for demonstration purposes.")
    with open('queries.txt', 'w', encoding='utf-8') as f:
        # Dictionary words (Cyrillic examples)
        f.write("привет\nмир\nпитон\nкод\nпроверка\n")
        # Queries to be processed
        f.write("привет\n")      # correct
        f.write("мр\n")          # 1 edit from мир
        f.write("питонн\n")      # 1 edit from питон
        f.write("проврка\n")     # 2 edits from проверка
        f.write("xyzabc\n")      # >2 edits away
        # Pad file to match user's original range values
        for i in range(100000):
             f.write(f"слово{i}\n")


try:
    # --- OPTIMIZATION: Read the file only ONCE ---
    print("Reading file into memory...")
    with open('dict.txt', 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    num_dictionary_words = 62_027

    # --- Step 1: Build the dictionary set from the first part of the file.
    print("Building dictionary...")
    dictionary = set(lines[:num_dictionary_words])
    print("Dictionary loaded.")

    # --- Step 2: Process the queries from the rest of the file.
    with open('queries.txt', 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    queries = lines
    ans = []

    print(f"Processing {len(queries)} queries...")
    for word in queries:
        # Case 1: The word is already correct.
        if word in dictionary:
            ans.append(f'{word} 0')
            continue

        # Call the on-the-fly correction finder
        path = find_correction(word, dictionary)

        # Case 2: A correction was found.
        if path:
            distance = len(path) - 1
            path_str = ' '.join(path[1:])
            ans.append(f'{word} {distance} {path_str}')
            continue

        # Case 3: No correction found within 2 edits.
        ans.append(f'{word} 3+')

    # --- Step 3: Write results to file.
    with open('ans.txt', 'w', encoding='utf-8') as f:
        for line in ans:
            f.write(line + '\n')

    print("Correction process finished. Check the 'ans.txt' file for results.")

except FileNotFoundError:
    print("Error: 'queries.txt' not found. Please ensure the file exists in the same directory.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


## 161. Генератор точек в круге

https://coderun.yandex.ru/problem/points-in-circle

In [ ]:
import sys

def main():

    def classifier(seq):
        cnt1, cnt2 = 0, 0
        for x, y in seq:
            if x ** 2 + y ** 2 > 0.25:
                cnt1 += 1
            else:
                cnt2 += 1

        return 1 if abs(cnt1 - cnt2) <= 100 else 2

    for _ in range(100):
        line = list(map(float, input().split()))
        points = [(x, y) for x, y in zip(line[::2], line[1::2])]
        print(classifier(points))

if __name__ == '__main__':
    main()

## 163. Сумма на гранях

https://coderun.yandex.ru/problem/ml-dice/description

In [ ]:
import sys


def main():
    from collections import Counter

    a = Counter(map(int, input().split()))
    k = int(input())
    first, not_first = sum(v * freq / 6 for v, freq in a.items()), sum(v * (1 - freq / 6) * freq / 6 for v, freq in a.items())
    expectation = first + (k - 1) * not_first


    print(expectation)


if __name__ == '__main__':
    main()



## 171. Коллекция статуэток

https://coderun.yandex.ru/problem/collection-of-figurines

In [ ]:
import sys


def main():
    from math import inf
    from collections import defaultdict, Counter

    n, k = map(int, input().split())
    a = list(map(int, input().split()))
    required = set(range(1, k + 1))
    cnt = Counter()
    cur_cost, min_cost, l = 0, inf, 0
    for r in range(n + 1):
        while not required:
            min_cost = min(min_cost, cur_cost)
            cnt[a[l]] -= 1
            cur_cost -= a[l]
            if not cnt[a[l]]:
                del cnt[a[l]]
                if a[l] <= k: required.add(a[l])
            l += 1

        if r < n:
            cnt[a[r]] += 1
            cur_cost += a[r]
            if a[r] in required:
                required.discard(a[r])

    print(min_cost)

if __name__ == '__main__':
    main()


## Решающий пень

https://coderun.yandex.ru/problem/stump/description

In [ ]:
import sys
from collections import defaultdict

def main():

    n = int(input())
    xa, ya = [], []
    s = 0
    err = 0
    for i in range(n):
        x, y = map(int, input().split())
        xa.append((x, i))
        ya.append(y)
        s += y

    s1, s2 = 0, s / n

    for i in range(n):
        err += (s / n - ya[i]) ** 2

    if len(xa) == 1:
        print(0, ya[0], xa[0][0])
        return

    xa.sort()

    a, b, c = s1, s2, xa[0][0]
    l, r = 0, err
    min_err = err / n
    cur_n = 0
    for i in range(n - 1):
        y = ya[xa[i][1]]
        s1 = (cur_n * s1 + y) / (cur_n + 1)
        l += (cur_n + 1) * (s1 - y) ** 2 / cur_n if cur_n else 0
        s2 = ((n - cur_n) * s2 - y) / (n - cur_n - 1)
        r -=  (n - cur_n - 1) * (s2 - y) ** 2 / (n - cur_n)
        cur_n += 1

        if xa[i + 1][0] - xa[i][0] and min_err > (l + r) / n:
            a, b, c = s1, s2, xa[i + 1][0]
            min_err = (l + r) / n

    print(a, b, c)



if __name__ == '__main__':
    main()

## Разминка

https://coderun.yandex.ru/problem/warm-up

In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import os
import zipfile
from typing import List

def download_data(public_key="https://yadi.sk/d/DZNVMv2ER3zs5A") -> List[str]:
    """
    Downloads data from a Yandex Disk public link.

    It first attempts to download a single file if the filename is specified
    in the response headers. If not, it saves the content and assumes it's a
    zip archive. If the zip assumption fails, it then inspects the content
    to treat it as a CSV or TXT file.

    Returns:
        List[str]: A list of file paths for the downloaded/extracted files,
                   or an empty list if an error occurs.
    """
    # public_key = "https://yadi.sk/d/DZNVMv2ER3zs5A" # Replace with the correct public key if needed
    base_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key="

    try:
        # Get the download link
        final_url = base_url + public_key
        response = requests.get(final_url)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
        download_info = response.json()

        if 'href' in download_info:
            download_url = download_info['href']

            # Download the file(s)
            download_response = requests.get(download_url, stream=True)
            download_response.raise_for_status()

            # Check for 'content-disposition' header for a single file name
            content_disposition = download_response.headers.get('content-disposition')
            if content_disposition and 'filename=' in content_disposition:
                filename = content_disposition.split('filename=')[1].strip('"')
                file_path = filename
                with open(file_path, 'wb') as f:
                    for chunk in download_response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded single file: {filename}")
                return [file_path] # Return a list with the single file path
            else:
                # If no filename, save to a temporary file first without an extension
                temp_file_path = 'downloaded_data.tmp'
                with open(temp_file_path, 'wb') as f:
                     for chunk in download_response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded content to temporary file: {temp_file_path}")

                # First, try to handle it as a zip file
                try:
                    with zipfile.ZipFile(temp_file_path, 'r') as zip_ref:
                        extracted_files = zip_ref.namelist()
                        zip_ref.extractall('.')
                        print(f"Successfully extracted {len(extracted_files)} files from archive.")
                    os.remove(temp_file_path) # Remove the temp file after extraction
                    print(f"Removed temporary archive: {temp_file_path}")
                    return extracted_files

                # --- NEW LOGIC: Fallback for non-zip files ---
                except zipfile.BadZipFile:
                    print(f"'{temp_file_path}' is not a zip file. Attempting to handle as a text file.")

                    # Inspect the first 1024 bytes to guess the file type
                    new_extension = '.txt' # Default to .txt
                    try:
                        with open(temp_file_path, 'r', encoding='utf-8') as f:
                            # A simple heuristic: if a comma is in the first line, it's likely a CSV
                            first_line = f.readline()
                            if ',' in first_line:
                                new_extension = '.csv'
                    except (UnicodeDecodeError, IOError) as e:
                        print(f"Could not read file as text to determine type: {e}. Defaulting to .txt")

                    # Rename the file with the determined extension
                    final_filename = 'downloaded_file' + new_extension
                    if os.path.exists(final_filename):
                        os.remove(final_filename) # Remove if it already exists to avoid errors
                    os.rename(temp_file_path, final_filename)

                    print(f"Handled download as a single file and renamed to: {final_filename}")
                    return [final_filename]

                except Exception as e:
                    print(f"An unexpected error occurred during file handling: {e}")
                    if os.path.exists(temp_file_path):
                        os.remove(temp_file_path) # Clean up on other errors
                    return []

        elif 'error' in download_info:
            print(f"Error from Yandex Disk API: {download_info['message']}")
            return []
        else:
            print("Unexpected response format from Yandex Disk API.")
            return []

    except requests.exceptions.RequestException as e:
        print(f"Error during network request: {e}")
        return []

# Example of how to use the function
if __name__ == '__main__':
    downloaded_files = download_data()
    if downloaded_files:
        print("\n--- Function finished ---")
        print(f"Final file paths returned: {downloaded_files}")
        # You can now proceed to load the data, e.g.:
        # if downloaded_files[0].endswith('.csv'):
        #     df = pd.read_csv(downloaded_files[0])
        #     print("\nSuccessfully loaded data into pandas DataFrame:")
        #     print(df.head())
    else:
        print("\n--- Function finished ---")
        print("Could not retrieve any files.")

Downloaded content to temporary file: downloaded_data.tmp
Successfully extracted 1 files from archive.
Removed temporary archive: downloaded_data.tmp

--- Function finished ---
Final file paths returned: ['train.tsv']


In [ ]:
if __name__ == '__main__':
    downloaded_files = download_data(public_key="https://yadi.sk/d/R4A1-cSGYddnZg")
    if downloaded_files:
        print("\n--- Function finished ---")
        print(f"Final file paths returned: {downloaded_files}")
        # You can now proceed to load the data, e.g.:
        # if downloaded_files[0].endswith('.csv'):
        #     df = pd.read_csv(downloaded_files[0])
        #     print("\nSuccessfully loaded data into pandas DataFrame:")
        #     print(df.head())
    else:
        print("\n--- Function finished ---")
        print("Could not retrieve any files.")

Downloaded content to temporary file: downloaded_data.tmp
Successfully extracted 1 files from archive.
Removed temporary archive: downloaded_data.tmp

--- Function finished ---
Final file paths returned: ['test.tsv']


In [ ]:
import sys


def main():
    from math import sin, cos, tan, sqrt, log
    from sklearn.linear_model import LinearRegression
    import numpy as np
    import pandas as pd

    data = pd.read_csv('train.tsv', sep='\t', header=None)
    n = 101
    X, y = [], []
    for i in range(n):
        row = list(map(float, data.iloc[i]))
        x, f_x = list(row[:-1]), row[-1]
        X.append(x)
        y.append(f_x)

    X = np.array(X)
    y = np.array(y)
    model = LinearRegression(fit_intercept=True)
    model.fit(X, y)
    test = pd.read_csv('test.tsv', sep='\t', header=None)
    test = np.array(test)
    ans = model.predict(test)
    with open('ans.txt', 'w') as f:
        for a in ans:
            f.write(str(a) + '\n')

if __name__ == '__main__':
    main()

## Обобщённый AUC

https://coderun.yandex.ru/problem/generalized-auc

In [ ]:
import sys

class FenwickTree:
    def __init__(self, n):
        self.n = n
        self.tree = [0.] * (n + 1)

    def update(self, i, delta):
        # i is 1-indexed in BIT
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i

    def query(self, i):
        s = 0.
        while i > 0:
            s += self.tree[i]
            i -= i & -i
        return s

    def range_query(self, l, r):
        return self.query(r) - self.query(l - 1)


def main():
    from bisect import bisect_right
    from math import inf

    def compress_values(values):
        unique = sorted(set(values))
        return {val: idx + 1 for idx, val in enumerate(unique)}

    n = int(input())
    arr = []
    b_vals = []

    for _ in range(n):
        t, y = map(float, input().split())
        arr.append((t, y))
        b_vals.append(y)

    arr.sort()

    auc_denum = 0.
    for j in range(n):
        i = bisect_right(arr, (arr[j][0], inf))
        auc_denum += max(n - i, 0)

    comp = compress_values(b_vals)
    comp_size = len(comp)
    bit = FenwickTree(comp_size)

    auc_num = 0.
    idx = 0

    while idx < n:
        current_a = arr[idx][0]
        group = []

        while idx < n and arr[idx][0] == current_a:
            group.append(arr[idx])
            idx += 1

        for a_val, b_val in group:
            pos = comp[b_val]
            count_less = bit.query(pos - 1)
            count_total = bit.query(pos)
            count_equal = count_total - count_less
            auc_num += count_less + 0.5 * count_equal

        for a_val, b_val in group:
            pos = comp[b_val]
            bit.update(pos, 1)


    print(auc_num / auc_denum)




if __name__ == '__main__':
    main()

## Restaurants

https://coderun.yandex.ru/problem/restaurants

https://coderun.yandex.ru/problem/restaurants-2

In [ ]:
if __name__ == '__main__':
    downloaded_files = download_data(public_key="https://yadi.sk/d/rm5AqkALDE19wQ")
    if downloaded_files:
        print("\n--- Function finished ---")
        print(f"Final file paths returned: {downloaded_files}")
        # You can now proceed to load the data, e.g.:
        # if downloaded_files[0].endswith('.csv'):
        #     df = pd.read_csv(downloaded_files[0])
        #     print("\nSuccessfully loaded data into pandas DataFrame:")
        #     print(df.head())
    else:
        print("\n--- Function finished ---")
        print("Could not retrieve any files.")

Downloaded content to temporary file: downloaded_data.tmp
Successfully extracted 1 files from archive.
Removed temporary archive: downloaded_data.tmp

--- Function finished ---
Final file paths returned: ['restaurants_train.txt']


In [ ]:
import numpy as np

# Load the data from the text file
r_i = []
d_i = []
r_j = []
d_j = []
r_aver = 0
cnt = 0

with open('/content/restaurants_train.txt', 'r') as f:
    for line in f:
        numbers = list(map(float, line.split()))
        if numbers[0] == 0:
            if numbers[1] > 0:
                r_aver += numbers[1]
                cnt += 1
            if numbers[2] > 0:
                r_aver += numbers[2]
                cnt += 1
            r_j.append(numbers[1] if numbers[1] > 0 else 6)
            r_i.append(numbers[2] if numbers[2] > 0 else 6)
            d_j.append(numbers[3])
            d_i.append(numbers[4])
        elif numbers[0] == 1:
            if numbers[1] > 0:
                r_aver += numbers[1]
                cnt += 1
            if numbers[2] > 0:
                r_aver += numbers[2]
                cnt += 1
            r_i.append(numbers[1] if numbers[1] > 0 else 6)
            r_j.append(numbers[2] if numbers[2] > 0 else 6)
            d_i.append(numbers[3])
            d_j.append(numbers[4])

# Convert lists to NumPy arrays for the optimization function
r_i = np.array(r_i)
d_i = np.array(d_i)
r_j = np.array(r_j)
d_j = np.array(d_j)

print("Data loaded and processed.")
print(f"Length of r_i: {len(r_i)}")
print(f"Length of d_i: {len(d_i)}")
print(f"Length of r_j: {len(r_j)}")
print(f"Length of d_j: {len(d_j)}")
print(f"Average rating for r_i: {r_aver / cnt}")
# print(f"Average rating for r_j: {r_j_aver / cnt_j}")

Data loaded and processed.
Length of r_i: 876
Length of d_i: 876
Length of r_j: 876
Length of d_j: 876
Average rating for r_i: 7.942749981439804


In [ ]:
import numpy as np
from scipy.optimize import minimize


def objective_function(params, r_i, d_i, r_j, d_j):
    w_r, w_d = params
    score_i = w_r * np.log(250. + r_i) - w_d * d_i ** (1 / 10)
    score_j = w_r * np.log(250. + r_j) - w_d * d_j ** (1 / 10)

    diff_scores = score_i - score_j

    log_sum = np.sum(np.log(1 + np.exp(diff_scores)))

    N = len(diff_scores)
    m = (1 / N) * log_sum
    return m

# Initial guess for w_r and w_d
initial_params = [0.5, 10]

# Perform the optimization using 'BFGS' method
result = minimize(objective_function, initial_params, args=(r_i, d_i, r_j, d_j), method='BFGS')

# Perform the optimization using 'Nelder-Mead' method (commented out for comparison)
# result = minimize(objective_function, initial_params, args=(r_i, d_i, r_j, d_j), method='Nelder-Mead')


# Print the results
print("Optimization results:")
print(f"Success: {result.success}")
print(f"Message: {result.message}")
print(f"Optimal parameters (w_r, w_d): {result.x}")
print(f"Minimum value of m: {result.fun}")

Optimization results:
Success: True
Message: Optimization terminated successfully.
Optimal parameters (w_r, w_d): [168.33046464  19.86933268]
Minimum value of m: 0.5435305366848623


In [ ]:
import sys
from math import log


def main():

    n = int(input())

    for _ in range(n):
        r, d = map(float, input().split())
        if r < 0:
            r = 6.0

        print(168.33046464 * log(250 + r) - 19.8693326 * d ** (1 / 10))


if __name__ == '__main__':
    main()

## Линейно разделимая выборка
https://coderun.yandex.ru/problem/linear-separability-problem/description

In [ ]:
import sys


def main():
    from sklearn.svm import LinearSVC
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    import numpy as np


    model = Pipeline([('scaler', StandardScaler()),
                    ('svm', LinearSVC(fit_intercept=False, random_state=0, tol=1e-5))])

    n, m = map(int, input().split())
    X, y = [], []
    for _ in range(n):
        x = list(map(float, input().split()))
        X.append(x[:m])
        y.append(1 if x[m] == 1. else 0)

    if len(set(y)) == 1:
        X, y = np.array(X), np.array([1 if y[0] else -1] * n)
        print(*np.linalg.lstsq(X, y, rcond=None)[0])
    else:
        model.fit(X, y)
        print(*model.named_steps['svm'].coef_[0])




if __name__ == '__main__':
    main()

## Уникальные запросы
https://coderun.yandex.ru/problem/unique-queries

In [ ]:
import sys
import hashlib
import string
import struct
import math

class SimpleHLL:
    def __init__(self, p):
        self.p = p
        self.m = 1 << p  # m = 2**p
        self.registers = [0] * self.m
        self.alpha = self._get_alpha(p, self.m)

    def _get_alpha(self, p, m):
        if p == 4:
            return 0.673
        if p == 5:
            return 0.697
        if p == 6:
            return 0.709
        return 0.7213 / (1 + 1.079 / m)

    def add(self, value):
        h = hashlib.sha256(value.encode('utf8')).digest()
        x = struct.unpack('<Q', h[:8])[0]

        j = x & (self.m - 1)
        w = x >> self.p

        rho = 64 - self.p - w.bit_length() + 1 if w > 0 else 64 - self.p
        self.registers[j] = max(self.registers[j], rho)

    def estimate(self):
        Z = float(sum([2.0**(-reg) for reg in self.registers]))
        E = self.alpha * (self.m**2) / Z
        if E <= 2.5 * self.m:
            V = self.registers.count(0)
            if V > 0:
                return self.m * math.log(self.m / float(V))

        two_to_64 = 2**64
        if E > (1/30.0) * two_to_64:
            return -two_to_64 * math.log(1.0 - E / two_to_64)

        return E

def main():

    n = int(input())
    precision = 12
    hll_scratch = SimpleHLL(p=precision)

    for _ in range(n):
        hll_scratch.add(input())

    estimated_count = hll_scratch.estimate()

    print(estimated_count)


if __name__ == '__main__':
    main()

## Coins
https://coderun.yandex.ru/problem/coins

https://gemini.google.com/app/e968da7fddc5f35c

In [ ]:
if __name__ == '__main__':
    downloaded_files = download_data(public_key="https://disk.yandex.com/d/N40N52wF5FIp-A")
    if downloaded_files:
        print("\n--- Function finished ---")
        print(f"Final file paths returned: {downloaded_files}")
        # You can now proceed to load the data, e.g.:
        # if downloaded_files[0].endswith('.csv'):
        #     df = pd.read_csv(downloaded_files[0])
        #     print("\nSuccessfully loaded data into pandas DataFrame:")
        #     print(df.head())
    else:
        print("\n--- Function finished ---")
        print("Could not retrieve any files.")

Downloaded content to temporary file: downloaded_data.tmp
Successfully extracted 1 files from archive.
Removed temporary archive: downloaded_data.tmp

--- Function finished ---
Final file paths returned: ['coins.in']


In [ ]:
with open('coins.in', 'r') as f:
    n = int(f.readline().strip())
    arr = []
    for i in range(n):
        k, m = map(int, f.readline().strip().split())
        arr.append(((m + 1) / (k + 2), i))

    arr.sort()

with open('coins.out', 'w') as f:
    for i in arr:
        f.write(str(i[1]) + '\n')


## Лента рекомендаций

https://coderun.yandex.ru/problem/recommendations/description

In [ ]:
import sys

class ListNode:

    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next



def main():
    from heapq import heappush, heappop
    n, m = map(int, input().split())
    x = []
    head = ListNode(val=0)
    cur = head
    for i in range(n):
        x.append(int(input()))
        if i < n - 1:
            cur.next = ListNode(val=i + 1)
            cur = cur.next

    ans = [0]
    cur = head
    while cur.next:
        if x[ans[-1]] != x[cur.next.val]:
            ans.append(cur.next.val)
            cur.next = cur.next.next
            cur = head
        else:
            cur = cur.next

    print(*ans)





if __name__ == '__main__':
    main()

## Sequence Clustering

https://coderun.yandex.ru/problem/sequence-clustering/description

In [ ]:
import sys
import numpy as np

def solve():

    N, K = map(int, input().split())

    alphabet_chars = sys.stdin.readline().strip()
    char_to_int = {char: i for i, char in enumerate(alphabet_chars)}
    stop_symbol_idx = K - 1
    char_to_int['S'] = stop_symbol_idx

    strings = [sys.stdin.readline().strip() for _ in range(N)]

    int_sequences = []
    for s in strings:
        if s:
            int_sequences.append([char_to_int[c] for c in s])

    if len(int_sequences) != N:
        return

    responsibilities = np.random.rand(N, 2)
    responsibilities /= responsibilities.sum(axis=1, keepdims=True)

    pi = np.zeros(2)
    q = np.zeros((2, K))
    P = np.zeros((2, K, K))

    smoothing_factor = 1e-9

    num_iterations = 25
    for _ in range(num_iterations):

        pi = responsibilities.sum(axis=0) / N

        q.fill(smoothing_factor)
        P.fill(smoothing_factor)

        for i in range(N):
            seq = int_sequences[i]
            r0, r1 = responsibilities[i, 0], responsibilities[i, 1]

            q[0, seq[0]] += r0
            q[1, seq[0]] += r1

            for j in range(len(seq) - 1):
                P[0, seq[j+1], seq[j]] += r0
                P[1, seq[j+1], seq[j]] += r1

            P[0, stop_symbol_idx, seq[-1]] += r0
            P[1, stop_symbol_idx, seq[-1]] += r1

        q_sum = q.sum(axis=1, keepdims=True)
        q /= q_sum

        P_sum = P.sum(axis=1, keepdims=True)
        P /= P_sum

        log_likelihoods = np.zeros((N, 2))

        log_pi = np.log(pi)
        log_q = np.log(q)
        log_P = np.log(P)

        for i in range(N):
            seq = int_sequences[i]

            ll0 = log_q[0, seq[0]]
            for j in range(len(seq) - 1):
                ll0 += log_P[0, seq[j+1], seq[j]]
            ll0 += log_P[0, stop_symbol_idx, seq[-1]]
            log_likelihoods[i, 0] = log_pi[0] + ll0

            ll1 = log_q[1, seq[0]]
            for j in range(len(seq) - 1):
                ll1 += log_P[1, seq[j+1], seq[j]]
            ll1 += log_P[1, stop_symbol_idx, seq[-1]]
            log_likelihoods[i, 1] = log_pi[1] + ll1

        max_log_ll = log_likelihoods.max(axis=1, keepdims=True)
        log_likelihoods -= max_log_ll
        likelihoods = np.exp(log_likelihoods)
        sum_likelihoods = likelihoods.sum(axis=1, keepdims=True)
        responsibilities = likelihoods / sum_likelihoods

    final_assignments = np.argmax(responsibilities, axis=1)

    for assignment in final_assignments:
        print(assignment)

if __name__ == "__main__":
    solve()

https://coderun.yandex.ru/problem/interpolation/description

In [ ]:
import sys


def main():
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import Pipeline
    import numpy as np


    model = Pipeline([('poly', PolynomialFeatures(degree=2)),
                    ('linear', LinearRegression(fit_intercept=False))])


    x = []
    test = []

    for i in range(1000):
        line = list(map(float, input().split()))
        x.append(line)

    for i in range(1000):
        line = list(map(float, input().split()))
        test.append(line)

    x = np.array(x)
    test = np.array(test)
    x, y = x[:, :-1], x[:, -1]
    model = model.fit(x, y)
    pred = model.predict(test)

    for e in pred:
        print(e)


if __name__ == '__main__':
    main()


In [ ]:
import sys


def main():
    from collections import defaultdict
    from heapq import heappush, heappop

    n = int(input())
    cov_positive = set([i for i, a in enumerate(map(int, input().split())) if a])
    meetings = defaultdict(list)
    graph = defaultdict(list)
    for i in range(n):
        x = list(map(int, input().split()))
        k, x = x[0], x[1:]
        for meet in x:
            for j in meetings[meet]:
                heappush(graph[i], (meet, j))
                heappush(graph[j], (meet, i))

            meetings[meet] += [i]

    seen = set()
    def dfs(person, meet):
        seen.add(person)
        for m, adj in sorted(graph[person]):
            if adj not in seen and meet <= m:
                dfs(adj, m)

    for person in cov_positive:
        dfs(person, 0)

    print(*[1 if i in seen else 0 for i in range(n)])



if __name__ == '__main__':
    main()

In [ ]:
import sys
from math import sqrt

def main():

    n = int(input())
    dist = 0
    stack = [(0, 0)]
    for _ in range(n):
        x, y = map(int, input().split())
        if y < stack[-1][1]:
            dist += sqrt((x - stack[-1][0])**2 + (y - stack[-1][1])**2)
            stack.append((x, y))
            continue

        while stack and y >= stack[-1][1]:
            dist += sqrt((x - stack[-1][0])**2 + (y - stack[-1][1])**2)
            stack.pop()

        stack += [(x, y)]


    print(dist)

if __name__ == '__main__':
    main()

In [ ]:
import sys


def main():
    n = int(input())
    y, y_inv = [], []
    s, s_inv = 0., 0.
    for _ in range(n):
        num = int(input())
        y.append(num)
        s += num
        y_inv.append((1 / num, num))
        s_inv += 1 / num

    y.sort()
    y_inv.sort()
    print(s / n)
    print(y[n // 2])
    s_cur = 0
    for j in range(n):
        s_cur += y_inv[j][0]
        if s_cur >= s_inv / 2:
            print(y_inv[j][1])
            break


if __name__ == '__main__':
    main()

## Исчезнувшая

https://coderun.yandex.ru/problem/in-search-of-kitty/description

In [ ]:
import sys


def main():
    n1, m1 = map(int, input().split())
    origin = []
    for _ in range(n1):
        origin.append(list(input()))

    n2, m2 = map(int, input().split())
    pict = []
    for _ in range(n2):
        pict.append(list(input()))

    o_rot90 = [list(row)[::-1] for row in zip(*origin)]
    o_rot180 = [row[::-1] for row in origin[::-1]]
    o_rot270 = [list(row) for row in zip(*origin)][::-1]

    def search(pict):
        for i in range(n2 - n1 + 1):
            for j in range(m2 - m1 + 1):
                picture = [row[j:j + m1] for row in pict[i:i + n1]]
                if picture == origin:
                    return True
                if picture == o_rot180:
                    return True

        for i in range(n2 - m1 + 1):
            for j in range(m2 - n1 + 1):
                picture = [row[j:j + n1] for row in pict[i:i + m1]]
                if picture == o_rot90:
                    return True
                if picture == o_rot270:
                    return True

        return False


    if search(pict):
        print("Yes")
    else:
        print("No")



if __name__ == '__main__':
    main()

## Rum
https://coderun.yandex.ru/problem/bottles-of-rum/description

In [ ]:
import sys

def main():
    from math import factorial
    from collections import Counter
    from itertools import chain, combinations

    s = input()
    words = set()
    x = 0
    for word in chain.from_iterable(combinations(s, r) for r in range(len(s) + 1)):
        cnt = tuple(sorted(Counter(word).items()))
        if cnt not in words:
            cur = factorial(len(word))
            words.add(cnt)
            for _, freq in cnt:
                cur //= factorial(freq)

            x += cur

    print(x - 1)



if __name__ == '__main__':
    main()

## Happiness

https://coderun.yandex.ru/problem/happiness/description

In [ ]:
import sys
from math import exp, log


def main():

    n, k = map(int, input().split())
    a, b = 0., 0.
    p = 0.
    for _ in range(n):
        x = list(map(float, input().split()))
        a += x[-2]
        b += x[-1]
        p += x[-1] * log(1. + (x[-2] / x[-1]))

    p = exp(p / b) - 1.
    print(f'{(a / b):.6f} {p:.6f} {(a / b):.6f}')


if __name__ == '__main__':
    main()

## Bandit

https://coderun.yandex.ru/problem/multiarmed-bandit/description

In [ ]:
import sys
import random

def find_max(arr):
    max_i, max_sample = -1, -1
    for i in range(len(arr)):
        sample = random.betavariate(arr[i][0], arr[i][1])

        if sample > max_sample:
            max_sample = sample
            max_i = i

    return max_i

def main(n, m, alpha, beta):

    machines = [[alpha, beta] for _ in range(m)]

    cur_machine = 1
    for _ in range(n):
        print(cur_machine, flush=True)
        res = int(input())
        if res:
            machines[cur_machine - 1][0] += 1.
        else:
            machines[cur_machine - 1][1] += 1.

        cur_machine = find_max(machines) + 1


if __name__ == '__main__':
    while True:
        n, m = map(int, input().split())
        if n == 0:
            break
        alpha, beta = map(float, input().split())
        main(n, m, alpha, beta)

## Gradient Ascent
https://coderun.yandex.ru/problem/gradient-free-optimization/description

In [ ]:
import sys
import math

class GradientAscentSolver:

    def __init__(self):
        self.x = 0.0
        self.y = 0.0
        self.h = 0.0

        self.query_count = 0
        self.max_queries = 200

        self.eps = 1e-7
        self.alpha = 10.0
        self.alpha_increase = 1.5 # Factor to increase step size on success
        self.alpha_decrease = 0.5 # Factor to decrease step size on failure

    def _query(self, qx, qy):
        qx = max(-50.0, min(100.0, qx))
        qy = max(-50.0, min(100.0, qy))

        print(f"{qx:.8f} {qy:.8f}", flush=True)
        self.query_count += 1

        response = sys.stdin.readline().strip().split()

        status = response[0]
        value = float(response[1])

        if status == '+':
            sys.exit(0)

        return value

    def solve(self):

        x0, y0, h0 = map(float, input().split())
        self.x, self.y, self.h = x0, y0, h0

        while self.query_count <= self.max_queries - 5:

            h_px = self._query(self.x + self.eps, self.y)
            h_nx = self._query(self.x - self.eps, self.y)
            h_py = self._query(self.x, self.y + self.eps)
            h_ny = self._query(self.x, self.y - self.eps)

            grad_x = (h_px - h_nx) / (2 * self.eps)
            grad_y = (h_py - h_ny) / (2 * self.eps)

            norm = math.sqrt(grad_x**2 + grad_y**2)

            if norm < 1e-12:
                break

            dir_x = grad_x / norm
            dir_y = grad_y / norm

            current_alpha = self.alpha
            found_better_point = False

            for _ in range(5):
                if self.query_count >= self.max_queries:
                    break

                x_new = self.x + current_alpha * dir_x
                y_new = self.y + current_alpha * dir_y

                h_new = self._query(x_new, y_new)

                if h_new > self.h:
                    self.x, self.y, self.h = x_new, y_new, h_new
                    self.alpha = current_alpha * self.alpha_increase
                    found_better_point = True
                    break
                else:
                    current_alpha *= self.alpha_decrease

            if not found_better_point:
                self.alpha = current_alpha

if __name__ == "__main__":
    solver = GradientAscentSolver()
    solver.solve()

## Distance Editing
https://coderun.yandex.ru/problem/distance-editing/description

In [ ]:
import sys


def main():
    from functools import cache

    n, m = map(int, input().split())
    s = input()
    t = input()
    ins, de, sw = map(int, input().split())
    dp = [[0 for _ in range(m + 1)] for _ in range(n + 1)]
    for i in range(1, m + 1):
        dp[0][i] = ins * i

    for i in range(1, n + 1):
        dp[i][0] = de * i

    for i in range(n):
        for j in range(m):
            dp[i + 1][j + 1] = min(dp[i][j + 1] + de, dp[i + 1][j] + ins, dp[i][j] + sw * (s[i] != t[j]))

    # cache
    # def dp(i, j):
    #     if i == 0:
    #       return ins * j
    #     if j == 0:
    #       return de * i

    #     return
    print(dp[n][m])


if __name__ == '__main__':
    main()

## Points Precision
https://coderun.yandex.ru/problem/points-precision

In [ ]:
from math import gcd

n = int(input())
k = (n * (n - 1)) // 2
pairs = {}
labels_p = {}
labels_v = {}
p, v = list(map(int, input().split())), list(map(int, input().split()))
cnt = 0
for i in range(n):
    cnt += i
    cnt -= labels_p.setdefault(p[i], 0) - pairs.setdefault((p[i], v[i]), 0)
    cnt -= labels_v.setdefault(v[i], 0) - pairs[(p[i], v[i])]
    labels_p[p[i]] += 1
    labels_v[v[i]] += 1
    pairs[(p[i], v[i])] += 1

g = gcd(cnt, k)
print(f'{cnt // g}/{k // g}')

## Concat

https://coderun.yandex.ru/problem/concatenation

In [ ]:
import sys


def main():
    from collections import defaultdict
    from itertools import product
    from heapq import heappush, heappop

    n = int(input())
    words = defaultdict(set)
    words_ = defaultdict(set)
    for i in range(1, n + 1):
        word = input()
        words[word].add(i)
        m = len(word)
        for j in range((m - 1) // 2):
            inner = word[j + 1:m - j - 1]
            if word[:j + 1] == word[m - j - 1:] and inner:
                words_[inner].add(i)

    pairs = []
    seen = set()
    for key, el in words.items():
        for i, j in product(el, words_[key]):
            if i != j and (i, j) not in seen:
                heappush(pairs, (i, j))
                heappush(pairs, (j, i))
                seen.add((i, j))
                seen.add((j, i))

    while pairs:
        print(*heappop(pairs))



if __name__ == '__main__':
    main()

## Bundle on Fingers
https://coderun.yandex.ru/problem/bundle-on-fingers/description

In [ ]:
import sys
import numpy as np

def main():

    n,m, k = map(int, input().split())
    a, c = [], []
    for _ in range(n):
        row = list(map(int, input().split()))
        a += [row]

    for _ in range(n - k + 1):
        row = list(map(int, input().split()))
        c += row

    a_ = []
    for i in range(n - k + 1):
        for j in range(m - k + 1):
            row = []
            for t in range(k):
                for l in range(k):
                    row += [a[i + t][j + l]]

            a_ += [row]

    a, c = np.array(a_), np.array(c)
    b = np.linalg.lstsq(a, c, rcond=None)[0]
    for i in range(k):
        row = [int(np.round(b[i * k + j])) for j in range(k)]
        print(*row)



if __name__ == '__main__':
    main()

## Profit
https://coderun.yandex.ru/problem/in-pursuit-of-profit

In [20]:
if __name__ == '__main__':
    downloaded_files = download_data(public_key="https://disk.yandex.ru/d/XNX_yGbysEvNLg")
    if downloaded_files:
        print("\n--- Function finished ---")
        print(f"Final file paths returned: {downloaded_files}")
        # You can now proceed to load the data, e.g.:
        # if downloaded_files[0].endswith('.csv'):
        #     df = pd.read_csv(downloaded_files[0])
        #     print("\nSuccessfully loaded data into pandas DataFrame:")
        #     print(df.head())
    else:
        print("\n--- Function finished ---")
        print("Could not retrieve any files.")

Downloaded content to temporary file: downloaded_data.tmp
'downloaded_data.tmp' is not a zip file. Attempting to handle as a text file.
Handled download as a single file and renamed to: downloaded_file.txt

--- Function finished ---
Final file paths returned: ['downloaded_file.txt']


In [21]:
import numpy as np

# import k-means
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

from collections import Counter

with open('downloaded_file.txt', 'r') as f:
    k, n, c, C = map(int, f.readline().split())
    pts = []
    for _ in range(n):
        pts += [list(map(int, f.readline().split()))]

    pts = np.array(pts)
    # apply k-means on pts array
    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=500, tol=1e-04, random_state=0)
    y_kmeans = kmeans.fit_predict(pts)
    centroids = kmeans.cluster_centers_
    d, cnt = Counter(), Counter(y_kmeans)
    groups = [[] for _ in range(k)]
    for i in range(n):
        d[y_kmeans[i]] += ((pts[i][0] - centroids[y_kmeans[i]][0]) ** 2 + (pts[i][1] - centroids[y_kmeans[i]][1]) ** 2) ** (1 / 8) + 1
        groups[y_kmeans[i]].append(i)
    for i in range(k):
        d[i] /= cnt[i]

    x = C - c * sum(d.values())
    print(x)
    for i in range(k):
        print(float(centroids[i][0]), float(centroids[i][1]))

    for i in range(k):
        print(*groups[i])



10170.007348771556
-53058703.980392165 51852296.50980388
35168793.15909094 -66400497.36363641
64729963.305555575 77221366.63888884
-54637256.36734694 -37287577.612244934
7276506.617021274 23056099.14893614
99763995.39473678 -51531695.39473688
60606427.47222226 25289284.027777757
-57108771.51111112 -91818839.88888894
-22514264.688888863 79304468.17777774
49413069.23076924 -26570667.403846193
-1562767.404255322 -25917317.957446843
-66633074.36956524 80466643.30434777
-66822597.9142857 12347010.6
65922168.66666666 -80891524.88095242
20764657.687500022 79665437.39583328
-7171150.725000002 -95405141.12500006
-29233822.270833295 -52849516.2291667
-82083823.1086957 -61939250.91304353
62454621.470588274 48097745.41176467
89695909.35135132 -6563493.621621621
-22272918.01694912 45759805.64406775
-23789753.487804852 10677984.073170733
98362489.42857136 75643596.59523804
7832519.6904761875 -55717894.09523813
-86622282.2 -35875107.43333336
19144656.700000014 63475927.57499996
93373855.01886785 4275

## Tricky Tree
https://coderun.yandex.ru/problem/tricky-tree/description

In [ ]:
import sys
import copy

def main():

    f, c, n = map(int, input().split())
    tree = [None] * (n + 1)

    for i in range(1, n + 1):
        p, q = map(int, input().split())

        line2 = sys.stdin.readline().strip()
        if p == -1 and q == -1:
            k = int(line2)
            tree[i] = {'type': 'leaf', 'class': k}
        else:
            s_str, t  = line2.split()
            t_val, s_val = float(t), int(s_str)
            tree[i] = {
                'type': 'internal',
                'left': p,
                'right': q,
                'threshold': t_val,
                'feature_idx': s_val
            }

    dataset = []

    def find_paths_to_leaves(node_idx, path_constraints):
        node = tree[node_idx]

        if node['type'] == 'leaf':
            sample = [0.0] * f

            for feature_idx, (lower_bound, upper_bound) in path_constraints.items():
                sample[feature_idx - 1] = lower_bound

            dataset.append(sample)
            return

        left_constraints = copy.deepcopy(path_constraints)
        feature = node['feature_idx']
        threshold = node['threshold']

        left_constraints[feature][1] = min(left_constraints[feature][1], threshold)
        find_paths_to_leaves(node['left'], left_constraints)

        right_constraints = copy.deepcopy(path_constraints)

        right_constraints[feature][0] = max(right_constraints[feature][0], threshold)
        find_paths_to_leaves(node['right'], right_constraints)


    initial_constraints = {i: [-1000.0, 1001.0] for i in range(1, f + 1)}

    find_paths_to_leaves(1, initial_constraints)

    print(len(dataset))
    for sample in dataset:
        print(" ".join(f"{val:.1f}" for val in sample))



if __name__ == '__main__':
    main()

## Beauty
https://coderun.yandex.ru/problem/beauty-sacrifice

In [ ]:
import sys
from bisect import bisect_left
from math import sqrt, inf

def dist(a, b):
    d = 0
    for x1, x2 in zip(a[1:], b[1:]):
        d += (x1 - x2) ** 2

    return sqrt(d)

def main():
    n, q = map(int, input().split())
    avail_ = {}
    to_ind = {}
    avail = []
    a = []
    for i in range(n):
        thing = list(map(int, input().split()))
        a += [thing[1:]]
        if thing[0]:
            avail += [thing[1:]]
            avail_[i] = thing[1]
            to_ind[len(avail) - 1] = i

    avail = [(e, i) for i, e in enumerate(avail)]
    avail.sort()
    for _ in range(q):
        i, k = map(int, input().split())
        if i in avail_ and avail_[i] <= k:
            print(1)
            print(i)
        else:
            print(5)
            j = bisect_left(avail, ([k] + [101] * 10, 0))
            x = avail[:j]
            x.sort(key=lambda e: dist(a[i], e[0]))
            print(*[to_ind[e] for _, e in x[:5]])



if __name__ == '__main__':
    main()

## Compression
https://coderun.yandex.ru/problem/compression-medal

In [ ]:
import sys
import numpy as np

def solve():

    n_str, m_str, p_str = sys.stdin.readline().split()
    n, m, p = int(n_str), int(m_str), float(p_str)
    data = []
    for _ in range(n):
        data += [list(map(float, input().split()))]

    data = np.array(data)
    X = data[:, :-1]
    y = data[:, -1]

    X_mean = np.mean(X, axis=0)
    y_mean = np.mean(y)

    X_std = np.std(X, axis=0)
    y_std = np.std(y)

    cov_xy = np.mean((X - X_mean) * (y[:, np.newaxis] - y_mean), axis=0)

    correlations = np.zeros(m)
    non_zero_std = (X_std * y_std) != 0
    correlations[non_zero_std] = cov_xy[non_zero_std] / (X_std[non_zero_std] * y_std)

    v = int(2 * m / 3)

    top_v_indices = np.argsort(np.abs(correlations))[::-1][:v]
    X_reduced = X[:, top_v_indices]

    print(v)
    for i in range(n):
        print(*list(X[i, :]))


solve()

## 1984
https://coderun.yandex.ru/problem/1984/description

In [ ]:
def calculate_z_array(s):

    n = len(s)

    z = [0] * n
    z[0] = n

    l, r = 0, 0

    for i in range(1, n):
        if i > r:
            l, r = i, i
            while r < n and s[r - l] == s[r]:
                r += 1
            z[i] = r - l
            r -= 1
        else:
            k = i - l

            if z[k] < r - i + 1:
                z[i] = z[k]
            else:

                l = i
                while r < n and s[r - l] == s[r]:
                    r += 1
                z[i] = r - l
                r -= 1

    return z

def search_pattern(text, pattern):

    concat_string = pattern + "$" + text
    z_array = calculate_z_array(concat_string)

    pattern_len = len(pattern)
    occurrences = []

    for i in range(len(z_array)):
        if z_array[i] == pattern_len:
            return True

    return False

def main():

    n, m = map(int, input().split())
    forbidden = []
    for _ in range(n):
        forbidden += [input()]

    for _ in range(m):
        x = input()
        if any(search_pattern(x, e) for e in forbidden):
            print('DELETE')
        else:
            print('KEEP')


if __name__ == '__main__':
    main()


## Fuzzies
https://coderun.yandex.ru/problem/fuzzies

In [ ]:
import sys


def main():

    n, t = map(int, input().split())
    a = list(map(int, input().split()))

    a.sort(reverse=True)
    cnt = 0
    while a and t >= a[-1]:
        t -= a.pop()
        cnt += 1



    print(cnt)



if __name__ == '__main__':
    main()

## cat search
https://coderun.yandex.ru/problem/cat-search

In [ ]:
import sys


class UnionFind:
    def __init__(self, n: int, vertices = None):
        if vertices:
            self.root = {v: v for v in vertices}
            self.size = {v: 1 for v in vertices}
            self.components = len(vertices)
        else:
            self.root = list(range(n + 1))
            self.size = [1] * (n + 1)
            self.components = n


    def find(self, x: int) -> int:
        if self.root[x] == x:
            return x
        self.root[x] = self.find(self.root[x])
        return self.root[x]


    def union(self, x: int, y: int) -> int:
        x = self.find(x)
        y = self.find(y)
        if x == y:
            return 0

        if self.size[x] > self.size[y]:
            self.size[x] += self.size[y]
            self.root[y] = x
        else:
            self.size[y] += self.size[x]
            self.root[x] = y

        self.components -= 1
        return 1

    def are_connected(self, x: int, y: int) -> bool:
        return self.find(x) == self.find(y)

def main():

    mat = []
    vert = {}
    cnt = 1
    for i, line in enumerate(sys.stdin.readlines()):
        row = list(map(int, line.split()))
        mat += [row]
        for j, e in enumerate(row):
            if row[j]:
                vert[(i, j)] = cnt
                cnt += 1


    n, m = len(mat), len(mat[0])
    uf = UnionFind(len(vert), vert)
    for i in range(n):
        for j in range(m):
            if mat[i][j]:
                if i < n - 1 and mat[i + 1][j]:
                    uf.union((i, j), (i + 1, j))
                if j < m - 1 and mat[i][j + 1]:
                    uf.union((i, j), (i, j + 1))

    x = uf.components
    print(x)
    for i in range(n):
        for j in range(m):
            if mat[i][j]:
                mat[i][j] = (vert[uf.find((i, j))])

        print(*mat[i])

if __name__ == '__main__':
    main()

## least real root
https://coderun.yandex.ru/problem/least-real-root/description

In [ ]:
import sys
import numpy as np
from numpy import linalg as LA
from math import inf, sqrt



def main():

    n = int(input())
    a, b = [], []
    for _ in range(n):
        row = list(map(float, input().split()))
        a += [row]

    for _ in range(n):
        row = list(map(float, input().split()))
        b += [row]

    a, b = np.array(a), np.array(b)
    c = np.block([
        [np.zeros((n, n)), np.identity(n)],
        [-b,                -a]
    ])
    lamb, _ = LA.eig(c)
    min_root = inf
    for lamb_ in lamb:
        if lamb_.imag != 0:
            continue

        min_root = min(min_root, lamb_.real)

    print(np.round(min_root, 4))


if __name__ == '__main__':
    main()

## feed
https://coderun.yandex.ru/problem/recomendation-feed/description

In [ ]:
import sys
from collections import Counter


def main():

    p, n, k = map(int, input().split())
    cnt = Counter()
    al = []
    for _ in range(p):
        s = input()
        al += [s]

    ids = list(map(int, input().split()))
    j = 0
    while j < p and n:
        cnt[al[j]] += 1
        if cnt[al[j]] <= k:
            print(al[j], '#' + str(ids[j]))
            n -= 1
        j += 1




if __name__ == '__main__':
    main()

## Mountain Trip
https://coderun.yandex.ru/problem/mountain-trip

In [ ]:
import sys

def feasible(x, s, ts):
    for i in range(len(ts) - s):
        if ts[i + s] - ts[i] <= x:
            return False

    return True

def main():

    n, s = map(int, input().split())
    ts = []
    for _ in range(n):
        ts += [int(input())]

    ts.sort()
    if n <= s:
        print('INF')
        return

    left, right = 0, ts[-1] - ts[0]
    while left <= right:
        mid = (left + right) // 2
        if feasible(mid, s, ts):
            left = mid + 1
        else:
            right = mid - 1

    if left > 0:
        print(left)
    else:
        print('Impossible')



if __name__ == '__main__':
    main()

## Antipigeon
https://coderun.yandex.ru/problem/antipigeon/description

https://gemini.google.com/app/4ee0498697087496

## Summer
https://coderun.yandex.ru/problem/summer-at-grandmas/description

In [ ]:
import sys
from collections import Counter
from math import isqrt

mod = 1_000_000_007

primes = [2, 3, 5, 7, 11, 13, 17, 19]

def exp(base, exponent):
   result = 1
   base %= mod
   while exponent > 0:
      if exponent % 2 == 1:
         result = (result * base) % mod

      exponent //= 2
      base = (base * base) % mod

   return result

def prime_factors(n):
   if n < 2:
      return 0

   factors = []

   while n % 2 == 0:
      factors.append(2)
      n //= 2

   for i in range(3, isqrt(n) + 1, 2):
      while n % i == 0:
         factors.append(i)
         n //= i

   if n > 2:
      factors.append(n)

   factors = Counter(factors)
   x = 0
   for i, p in enumerate(primes):
      if factors[p] % 2:
         x += (1 << i)

   return x

def character(v, k):
   it = 0
   for i in range(1, k + 1):
      x = prime_factors(i)
      if (x & v).bit_count() % 2:
         it -= 1
      else:
         it += 1

   return it

def main():

   m, k = map(int, input().split())

   d = 8
   for i, p in enumerate(primes):
      if p > k:
         d = i + 1
         break

   ans = 0 #
   for v in range(2 ** d):
      ans += exp(character(v, k), m)
      ans %= mod

   ans *= exp(2 ** d, mod - 2) * exp(exp(k, m), mod - 2)
   ans %= mod
   print(ans)



if __name__ == '__main__':
    main()

## 1984 2.0
https://coderun.yandex.ru/problem/1984-2/description?compiler=cpp

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <queue>
#include <map>

struct TrieNode {
    std::map<unsigned char, TrieNode*> children;
    TrieNode* failureLink;
    bool has_output;

    TrieNode() : failureLink(nullptr), has_output(false) {}
};

class AhoCorasick {
private:
    TrieNode* root;
    std::vector<TrieNode> node_pool;
    size_t nodes_count;

    TrieNode* newNode() {
        if (nodes_count < node_pool.size()) {
            return &node_pool[nodes_count++];
        }
        return nullptr;
    }

    TrieNode* getGo(TrieNode* node, unsigned char ch) {
        if (node->children.count(ch)) {
            return node->children.at(ch);
        }

        if (node == root) {
            return root;
        }

        return node->children[ch] = getGo(node->failureLink, ch);
    }

public:
    AhoCorasick(const std::vector<std::string>& patterns) {
        size_t total_len = 1;
        for (const auto& p : patterns) {
            total_len += p.length();
        }

        node_pool.resize(total_len);
        nodes_count = 0;
        root = newNode();

        buildTrie(patterns);
        buildAutomaton();
    }

    void buildTrie(const std::vector<std::string>& patterns) {
        for (const auto& pattern : patterns) {
            TrieNode* currentNode = root;
            for (unsigned char ch : pattern) {
                if (currentNode->children.find(ch) == currentNode->children.end()) {
                    currentNode->children[ch] = newNode();
                }
                currentNode = currentNode->children.at(ch);
            }
            currentNode->has_output = true;
        }
    }

    void buildAutomaton() {
        std::queue<TrieNode*> q;
        root->failureLink = root;
        q.push(root);

        while (!q.empty()) {
            TrieNode* currentNode = q.front();
            q.pop();

            std::vector<unsigned char> keys;
            for(auto const& [key, val] : currentNode->children) {
                keys.push_back(key);
            }

            for(unsigned char ch : keys) {
                TrieNode* child = currentNode->children.at(ch);

                if (currentNode == root) {
                    child->failureLink = root;
                } else {
                    child->failureLink = getGo(currentNode->failureLink, ch);
                }

                child->has_output = child->has_output || child->failureLink->has_output;
                q.push(child);
            }
        }
    }

    bool checkString(const std::string& text) {
        TrieNode* currentNode = root;
        for (unsigned char ch : text) {
            currentNode = getGo(currentNode, ch);
            if (currentNode->has_output) {
                return true;
            }
        }
        return false;
    }
};


int main() {
    std::ios_base::sync_with_stdio(false);
    std::cin.tie(NULL);

    int n, m;
    std::cin >> n >> m;
    std::cin.ignore();

    std::vector<std::string> forbidden(n);
    for (int i = 0; i < n; ++i) {
        std::getline(std::cin, forbidden[i]);
    }

    AhoCorasick trie(forbidden);

    for (int i = 0; i < m; ++i) {
        std::string text;
        std::getline(std::cin, text);
        if (trie.checkString(text)) {
            std::cout << "DELETE\n";
        } else {
            std::cout << "KEEP\n";
        }
    }

    return 0;
}